# UC-11 — silver (Fabric)

Leest de zeven `bronze_*` Delta-tabellen en schrijft `silver_*` Delta-tabellen
met dezelfde kolommen die `int_klantreis_events` verwacht. Deduplicatie en
expliciete kolom-cast (rename `source_ts → ingestion_source_ts` voor persona)
zijn de enige transformaties — Fabric-batch-equivalent van de Stackable
staging-views die JSON-extractie uit Kafka-bronze deden.

**Idempotent**: `mode('overwrite')`.

In [ ]:
workspace_id = "878307a8-e99a-4b9d-91c4-7b8fc457183b"
lakehouse_id = "10af248d-ba98-48c6-94db-fd2289b4f4a2"

In [ ]:
from pyspark.sql.functions import col

TABLES_ROOT = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

def transform(bronze_name, silver_name, transform_fn):
    src = f"{TABLES_ROOT}/{bronze_name}"
    dst = f"{TABLES_ROOT}/{silver_name}"
    df = spark.read.format("delta").load(src)
    df_out = transform_fn(df).dropDuplicates()
    df_out.write.format("delta").mode("overwrite").save(dst)
    print(f"{silver_name}: {df_out.count()} rijen")

In [ ]:
# ─── silver_persona ─ rename source_ts naar ingestion_source_ts ─────
transform(
    "bronze_persona_created",
    "silver_persona",
    lambda df: df.select(
        col("bsn"),
        col("voornaam"),
        col("achternaam"),
        col("geslacht"),
        col("geboortedatum"),
        col("woonplaats"),
        col("source_ts").alias("ingestion_source_ts"),
        col("event_date"),
    ),
)

In [ ]:
# ─── silver_polisadm_ikv ─ pass-through na dedup ────────────────────
transform(
    "bronze_polisadm_ikv",
    "silver_polisadm_ikv",
    lambda df: df.select(
        col("ikv_id"),
        col("bsn"),
        col("werkgever_naam"),
        col("aanvang_dienstverband"),
        col("einde_dienstverband"),
        col("loon_bruto_jaar"),
    ),
)

In [ ]:
# ─── silver_ww_aanvraag ─────────────────────────────────────────────
transform(
    "bronze_ww_aanvraag",
    "silver_ww_aanvraag",
    lambda df: df.select(
        col("aanvraag_id"),
        col("bsn"),
        col("aanvraag_datum"),
        col("status"),
        col("reden_einde_dienstverband"),
    ),
)

In [ ]:
# ─── silver_zw_melding ──────────────────────────────────────────────
transform(
    "bronze_zw_melding",
    "silver_zw_melding",
    lambda df: df.select(
        col("melding_id"),
        col("bsn"),
        col("eerste_ziektedag"),
        col("duur_dagen"),
    ),
)

In [ ]:
# ─── silver_wia_aanvraag ────────────────────────────────────────────
transform(
    "bronze_wia_aanvraag",
    "silver_wia_aanvraag",
    lambda df: df.select(
        col("aanvraag_id"),
        col("bsn"),
        col("aanvraag_datum"),
        col("status"),
        col("onderdeel"),
        col("arbeidsongeschikt_pct"),
        col("regio_code"),
    ),
)

In [ ]:
# ─── silver_wajong_dossier ──────────────────────────────────────────
transform(
    "bronze_wajong_dossier",
    "silver_wajong_dossier",
    lambda df: df.select(
        col("dossier_id"),
        col("bsn"),
        col("ingangsdatum"),
        col("regime"),
        col("arbeidsvermogen"),
    ),
)

In [ ]:
# ─── silver_crm_contact ─────────────────────────────────────────────
transform(
    "bronze_crm_contact",
    "silver_crm_contact",
    lambda df: df.select(
        col("contact_id"),
        col("bsn"),
        col("contact_ts"),
        col("kanaal"),
        col("onderwerp"),
        col("duur_seconden"),
    ),
)

In [ ]:
print("Silver klaar.")